# Interactive Visualization with Plotly

Plotly is a Python library used for creating interactive visualizations (graphs & charts). Unlike Matplotlib and Seaborn which create static images, Plotly renders an HTML document and uses JavaScript under the hood to enable interactivity. Plotly also offers a large selection of chart types to choose from.

In [1]:
!pip install plotly -q

In [2]:
# @title
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

np.random.seed(42)

resolution_times = np.random.exponential(scale=4.0, size=500) + 0.5
df_history = pd.DataFrame({"Resolution_Hours": resolution_times})


resp_mins = np.random.uniform(1, 15, 120)
csat_scores = 100 - (resp_mins * 3.5) + np.random.normal(0, 5, 120)
csat_scores = np.clip(csat_scores, 10, 100)
df_scatter = pd.DataFrame({"Response_Minutes": resp_mins, "CSAT_Score": csat_scores})

hours = np.arange(0, 24)
incoming = 50 + 35 * np.sin(hours / 3) + np.random.normal(0, 5, 24)
resolved = 45 + 30 * np.sin((hours - 1) / 3) + np.random.normal(0, 4, 24)

df_lines = pd.DataFrame({
    "Hour": hours,
    "Incoming_Tickets": incoming,
    "Resolved_Tickets": resolved
})

departments = ["Billing", "Technical Support", "Account Access", "Integrations", "Sales Queries"]
ticket_counts = [1420, 2850, 890, 610, 1150]
df_categories = pd.DataFrame({"Department": departments, "Ticket_Count": ticket_counts})


fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Distribution of Resolution Times",
        "CSAT Score vs. First Response Time",
        "24-Hour Operational Flow",
        "Ticket Volume by Department"
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)


px_hist = px.histogram(df_history, x="Resolution_Hours", nbins=25)
px_hist.update_traces(marker_color="#0284c7", marker_line_color="#ffffff", marker_line_width=0.5)
fig.add_trace(px_hist.data[0], row=1, col=1)

px_scatter = px.scatter(df_scatter, x="Response_Minutes", y="CSAT_Score")
px_scatter.update_traces(marker=dict(size=8, color="#dc2626", opacity=0.7))
fig.add_trace(px_scatter.data[0], row=1, col=2)

px_line_in = px.line(df_lines, x="Hour", y="Incoming_Tickets")
px_line_in.update_traces(line=dict(color="#2563eb", width=3), name="Incoming")

px_line_res = px.line(df_lines, x="Hour", y="Resolved_Tickets")
px_line_res.update_traces(line=dict(color="#16a34a", width=3, dash="dash"), name="Resolved")

fig.add_trace(px_line_in.data[0], row=2, col=1)
fig.add_trace(px_line_res.data[0], row=2, col=1)

px_bar = px.bar(df_categories, x="Department", y="Ticket_Count")
px_bar.update_traces(marker_color="#7c3aed", width=0.6)
fig.add_trace(px_bar.data[0], row=2, col=2)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=800,
    showlegend=False,
    margin=dict(l=60, r=40, t=80, b=60)
)

for title in fig['layout']['annotations']:
    title['font'] = dict(size=14, color="#111827", family="Arial")

fig.update_xaxes(title_text="Hours to Close Ticket", row=1, col=1)
fig.update_yaxes(title_text="Number of Tickets", row=1, col=1)

fig.update_xaxes(title_text="First Response Time (Mins)", row=1, col=2)
fig.update_yaxes(title_text="CSAT Score (0-100)", row=1, col=2)

fig.update_xaxes(title_text="Hour of Day (24h)", row=2, col=1)
fig.update_yaxes(title_text="Ticket Count", row=2, col=1)

fig.update_xaxes(title_text="Department Tier", row=2, col=2)
fig.update_yaxes(title_text="Total Open Volume", row=2, col=2)

fig.update_xaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#e5e7eb", zeroline=False)

fig.show()

The following topics are covered in this tutorial:

- Creating figures & adding interactive elements
- A quick tour of popular interactive charts
- Using Plotly as a plotting backend for Pandas
- Creating and exploring 3D graphs
- Adding controls and animating graphs

## Creating Figures and Adding Interactive Elements

Like Matplotlib, Plotly provides several low-level functions for creating and customizing figures. While they offer fine grained control over various aspects of a graph, they're quite verbose and can be cumbersome to use. We'll start out by using Plotly express, a high-level API similar to Seaborn, that allows creating and customizing charts with a single line of code.

Plotly express is often imported using the alias `px`.

In [3]:
import plotly.express as px

Plotly Express Doc: https://plotly.com/python/plotly-express/

Let's download a [country-wise population dataset](https://data.worldbank.org/indicator/SP.POP.TOTL) from World Bank Open Data.

In [4]:
!wget -q https://api.worldbank.org/v2/en/indicator/SP.POP.TOTL?downloadformat=csv -O population.csv

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
population_csv_url = 'https://raw.githubusercontent.com/nawaz0x1/Quantropy/refs/heads/main/Data%20Visualization/data/population.csv'
population_df = pd.read_csv(population_csv_url, index_col = "Year")
population_df.head()

,Aruba,Africa Eastern and Southern,Afghanistan,Africa Western and Central,Angola,Albania,Andorra,Arab World,United Arab Emirates,Argentina,...,Virgin Islands (U.S.),Viet Nam,Vanuatu,World,Samoa,Kosovo,"Yemen, Rep.",South Africa,Zambia,Zimbabwe
Year,,,,,,,,,,,,,,,,,,,,,
1960,54922.0,130075728.0,9035043.0,97630925.0,5231654.0,1608800.0,9510.0,91540853.0,131334.0,20386045.0,...,32500.0,32531933.0,64431.0,3.021513e+09,112490.0,984846.0,5532301.0,16440172.0,3153729.0,3809389.0
1961,55578.0,133534923.0,9214083.0,99706674.0,5301583.0,1659800.0,10283.0,93931683.0,137989.0,20726276.0,...,34300.0,33409059.0,66264.0,3.062768e+09,115496.0,1011421.0,5655232.0,16908035.0,3254086.0,3930401.0
1962,56320.0,137171659.0,9404406.0,101854756.0,5354310.0,1711319.0,11086.0,96428599.0,144946.0,21072538.0,...,35000.0,34288560.0,68174.0,3.117372e+09,118597.0,1036950.0,5782221.0,17418522.0,3358099.0,4055959.0
1963,57002.0,140945536.0,9604487.0,104089175.0,5408320.0,1762621.0,11915.0,99038509.0,152211.0,21421705.0,...,39800.0,35249101.0,70159.0,3.184063e+09,121764.0,1062737.0,5911135.0,17954564.0,3465907.0,4185877.0
1964,57619.0,144904094.0,9814318.0,106388440.0,5464187.0,1814135.0,12764.0,101729760.0,159692.0,21769453.0,...,40800.0,36201563.0,72219.0,3.251253e+09,124894.0,1090270.0,6048006.0,18511361.0,3577017.0,4320006.0


In [6]:
px.line(population_df['Bangladesh'] , title="Population")

In [7]:
population_df.tail(5)

,Aruba,Africa Eastern and Southern,Afghanistan,Africa Western and Central,Angola,Albania,Andorra,Arab World,United Arab Emirates,Argentina,...,Virgin Islands (U.S.),Viet Nam,Vanuatu,World,Samoa,Kosovo,"Yemen, Rep.",South Africa,Zambia,Zimbabwe
Year,,,,,,,,,,,,,,,,,,,,,
2022,107310.0,731821393.0,40578842.0,497387180.0,35635029.0,2451636.0,79705.0,471352066.0,10074977.0,45407904.0,...,105413.0,99680655.0,313046.0,7.989545e+09,215261.0,1768096.0,38222876.0,62378410.0,20152938.0,16069056.0
2023,107359.0,750491370.0,41454761.0,509398589.0,36749906.0,2414095.0,80856.0,482105978.0,10483751.0,45538401.0,...,104917.0,100352192.0,320409.0,8.064058e+09,216663.0,1682668.0,39390799.0,63212384.0,20723965.0,16340822.0
2024,107995.0,769280888.0,42647492.0,521764076.0,37885849.0,2377128.0,81938.0,492612632.0,10986400.0,45696159.0,...,104377.0,100987686.0,327777.0,8.141809e+09,218019.0,1594353.0,40583164.0,64007187.0,21314956.0,16634373.0
2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unnamed: 67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Dropping the last 2 columns as they dont have any values 

In [8]:
population_df.drop(population_df.tail(2).index,axis=0,inplace=True)

In [9]:
population_df.tail(2)

,Aruba,Africa Eastern and Southern,Afghanistan,Africa Western and Central,Angola,Albania,Andorra,Arab World,United Arab Emirates,Argentina,...,Virgin Islands (U.S.),Viet Nam,Vanuatu,World,Samoa,Kosovo,"Yemen, Rep.",South Africa,Zambia,Zimbabwe
Year,,,,,,,,,,,,,,,,,,,,,
2023,107359.0,750491370.0,41454761.0,509398589.0,36749906.0,2414095.0,80856.0,482105978.0,10483751.0,45538401.0,...,104917.0,100352192.0,320409.0,8.064058e+09,216663.0,1682668.0,39390799.0,63212384.0,20723965.0,16340822.0
2024,107995.0,769280888.0,42647492.0,521764076.0,37885849.0,2377128.0,81938.0,492612632.0,10986400.0,45696159.0,...,104377.0,100987686.0,327777.0,8.141809e+09,218019.0,1594353.0,40583164.0,64007187.0,21314956.0,16634373.0


In [10]:
px.line(population_df['Bangladesh'] , title="Population")


In [13]:
px.line(population_df['Ireland'], title="Population")

Note the following:

* `px.line` automatically picks the index of the series as the X-axis.
* You can hover over any point on the line to view the exact value.
* You can Zoom in and out using the controls to take a closer look at specific areas of the chart.
* There are several other controls e.g pan, autoscale, download PNG etc.

For fine grained control over various aspects of the chart, we can use the Figure object returned by `px.line`. Let's change the axis labels, chart colors, and ensure that the y axis starts at 0.

In [11]:
fig=px.line(population_df['Bangladesh'])

In [12]:
# Set axis & legend labels
fig.update_layout(
    title="Year-Wise Population",
    xaxis_title="Year",
    yaxis_title="Population",
    legend_title="Country",
    plot_bgcolor='#dedddc',
    font=dict(
        family="Arial",
        size=14,
        color="#040d78"
    )
)

# Start the Y axis from 0
fig.update_yaxes(rangemode='tozero')

Here's a list of properties you can set using `update_layout`: https://plotly.com/python/reference/layout/

Plotly also has built-in support for Pandas dataframes.

In [13]:
europe_df = population_df[['Italy', 'Germany', 'Switzerland']]
europe_df.head()

,Italy,Germany,Switzerland
Year,,,
1960,50199700.0,72814900.0,5327827.0
1961,50536350.0,73377632.0,5434294.0
1962,50879450.0,74025784.0,5573815.0
1963,51252000.0,74714353.0,5694247.0
1964,51675350.0,75318337.0,5789228.0


In [14]:
fig = px.line(europe_df,
              title='Population',
              color_discrete_sequence=["green", "cornflowerblue", "red"]
              )

fig.update_layout(yaxis_title='Population',
                  legend_title='Countries',
                  font_size=14)

fig.update_yaxes(rangemode='tozero')

fig.show()

Note that apart from providing RGB hexcodes for colors, we can also use named CSS colors: https://www.w3schools.com/cssref/css_colors.php

Switching from a line chart to a bar chart is simply a matter of replacing `plt.line` with `plt.bar`.

In [15]:
px.bar(population_df[['Bangladesh', 'Pakistan']],
       title="Population",
       barmode='group'
       )

EXERCISE: Compare the annual population increase of India and China using a line chart. Which of the two countries is growing faster? Hint: Use population_df.diff()

In [16]:
px.line(population_df[['India','China']])

In [17]:
px.line(population_df[['India','China']].diff(),title="Population Growth")

> **EXERCISE**: Compare the populations of 10 most populous African countries over the last 50 years using line charts.

In [18]:
african_countries = [
    'Nigeria', 'Ethiopia', 'Egypt, Arab Rep.', 'Congo, Dem. Rep.',
    'South Africa', 'Tanzania', 'Kenya', 'Algeria', 'Uganda',
    'Sudan', 'Morocco', 'Angola', 'Ghana', 'Mozambique', 'Madagascar'
]

african_population = population_df[african_countries]
african_population_50 = african_population.dropna().tail(50)

In [19]:
african_population_50

,Nigeria,Ethiopia,"Egypt, Arab Rep.","Congo, Dem. Rep.",South Africa,Tanzania,Kenya,Algeria,Uganda,Sudan,Morocco,Angola,Ghana,Mozambique,Madagascar
Year,,,,,,,,,,,,,,,
1975,63410815.0,31723252.0,38875327.0,23021101.0,25690940.0,16138361.0,13511671.0,15678596.0,11725240.0,14064798.0,17150786.0,6842947.0,10294308.0,9540817.0,7739336.0
1976,65258009.0,32534418.0,39795557.0,23624031.0,26395450.0,16736211.0,13957623.0,16409170.0,12037294.0,14569716.0,17574182.0,7074664.0,10594913.0,9867209.0,7976831.0
1977,67234590.0,33167272.0,40748870.0,24229061.0,27118952.0,17352929.0,14428819.0,17015994.0,12359669.0,15086614.0,18014756.0,7317829.0,10911871.0,10224975.0,8222240.0
1978,69326532.0,33734592.0,41761431.0,24992689.0,27869507.0,17976566.0,14934126.0,17506974.0,12692113.0,15695003.0,18477385.0,7576734.0,11248450.0,10618250.0,8475864.0
1979,71498242.0,34238652.0,42818628.0,25886782.0,28634162.0,18592038.0,15460666.0,18036655.0,13002163.0,16355352.0,18955589.0,7847207.0,11597255.0,11028809.0,8737581.0
1980,73764641.0,34428514.0,43950413.0,26711099.0,29518857.0,19189769.0,16018459.0,18607174.0,13275741.0,17058905.0,19459943.0,8133872.0,11941448.0,11336221.0,9006032.0
1981,76068103.0,35304012.0,45147224.0,27500515.0,30541044.0,19784703.0,16601507.0,19220704.0,13556452.0,17780572.0,19985401.0,8435607.0,12281712.0,11568127.0,9279011.0
1982,78378701.0,36701290.0,46407611.0,28338190.0,31615339.0,20398571.0,17206175.0,19872348.0,13873631.0,18486645.0,20520318.0,8751648.0,12643819.0,11824370.0,9557671.0
1983,80438260.0,37740817.0,47786402.0,29250963.0,32739304.0,21079874.0,17846374.0,20558115.0,14221881.0,19109770.0,21060696.0,9082983.0,13029836.0,12073822.0,9843044.0


In [22]:
fig = px.line(african_population_50,
              title="Population of last 10 years of African countries",
              labels={'value':'Total Population', 'Year':'Year','variable':'Country'})

fig.show()

## A quick tour of popular interactive charts

Plotly express provides more than 30 figure for creating different types of figures. Let's explore some popular interactive visualization techniques. We'll use the [built-in datasets](https://plotly.com/python-api-reference/generated/plotly.express.data.html) from `px.data` to demonstrate their usage.

### Scatter Plot